# Catchment Climate Data Toolkit

**Interactive watershed delineation + catchment-area-weighted PRISM meteorology**

This Google Colab notebook:
1. hydrolocates an outlet coordinate to the USGS NHDPlus network;
2. delineates its upstream catchment;
3. retrieves the upstream stream network;
4. builds an interactive map;
5. extracts catchment-area-weighted daily PRISM precipitation and temperature; and
6. exports analysis-ready CSV and GeoJSON files.

> **Start with a short PRISM date range to verify the workflow.** Daily PRISM data are distributed as individual grids by date/variable, so decades of daily data require many downloads.


In [ ]:
!pip -q install geopandas folium requests shapely pyproj rasterio tqdm matplotlib


## 1. User settings
Change only this cell for a new catchment/date range.


In [ ]:
LAT = 40.605278
LON = -88.898889

# Start with a short test. Expand after confirming the result.
START_DATE = "2024-01-01"
END_DATE   = "2024-01-31"

PRISM_VARIABLES = ["ppt", "tmin", "tmax"]
UPSTREAM_DISTANCE_KM = 500


## 2. Imports and helper functions


In [ ]:
import os, io, shutil, zipfile, time
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

from shapely.geometry import shape, Point, box
from pyproj import CRS
import rasterio
from rasterio.transform import xy
from tqdm.auto import tqdm

NLDI = "https://api.water.usgs.gov/nldi/linked-data"
PRISM = "https://services.nacse.org/prism/data/get"

CACHE = Path("/content/prism_cache")
OUTPUTS = Path("/content/outputs")
CACHE.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "CatchmentClimateDataToolkit/1.0 research workflow"})

def get_json(url, timeout=180):
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


## 3. Hydrolocate the outlet


In [ ]:
hydro_url = f"{NLDI}/hydrolocation?coords=POINT({LON}%20{LAT})&f=json"
hydro = get_json(hydro_url)

if not hydro.get("features"):
    raise RuntimeError("NLDI could not hydrolocate this coordinate.")

feature = hydro["features"][0]
props = feature.get("properties", {})
COMID = str(props.get("comid") or props.get("identifier") or "")

if not COMID:
    raise RuntimeError(f"Could not identify an NHDPlus COMID. Returned properties: {props}")

geom = shape(feature["geometry"])
if geom.geom_type == "Point":
    snapped = geom
else:
    snapped = geom.interpolate(0.5, normalized=True)

print(f"Input outlet : {LAT:.6f}, {LON:.6f}")
print(f"NHDPlus COMID: {COMID}")
print(f"Hydrolocated : {snapped.y:.6f}, {snapped.x:.6f}")


## 4. Delineate upstream catchment

`splitCatchment=true` requests a more point-specific local catchment split when supported by NLDI.


In [ ]:
basin_url = f"{NLDI}/comid/{COMID}/basin?f=json&simplified=false&splitCatchment=true"
basin_json = get_json(basin_url)

if not basin_json.get("features"):
    raise RuntimeError("NLDI returned no basin polygon.")

basin = gpd.GeoDataFrame.from_features(basin_json["features"], crs="EPSG:4326")
basin = basin[basin.geometry.notna()].copy()
basin["geometry"] = basin.geometry.make_valid()

# Equal-area CRS for area reporting
basin_ea = basin.to_crs("EPSG:5070")
area_m2 = basin_ea.geometry.area.sum()

print(f"Catchment area: {area_m2/1e6:,.2f} km²")
print(f"Catchment area: {area_m2/10000:,.2f} ha")
print(f"Catchment area: {area_m2/4046.8564224:,.2f} acres")


## 5. Retrieve upstream stream network


In [ ]:
stream_url = (
    f"{NLDI}/comid/{COMID}/navigation/UT/flowlines"
    f"?distance={UPSTREAM_DISTANCE_KM}&f=json"
)

try:
    stream_json = get_json(stream_url)
    streams = (
        gpd.GeoDataFrame.from_features(stream_json["features"], crs="EPSG:4326")
        if stream_json.get("features") else None
    )
except Exception as e:
    streams = None
    print("Stream-network retrieval warning:", e)

print("Upstream flowlines:", 0 if streams is None else len(streams))


## 6. Interactive catchment map


In [ ]:
m = folium.Map(location=[LAT, LON], zoom_start=10, tiles=None)

folium.TileLayer("OpenStreetMap", name="Street Map").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite"
).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Topographic"
).add_to(m)

folium.GeoJson(
    basin.to_json(),
    name="Upstream Catchment",
    style_function=lambda _: {"color":"#0057b8","weight":3,"fillColor":"#3388ff","fillOpacity":0.18},
).add_to(m)

if streams is not None:
    folium.GeoJson(
        streams.to_json(),
        name="Upstream Streams",
        style_function=lambda _: {"color":"#00a6d6","weight":2,"opacity":0.9},
    ).add_to(m)

folium.Marker(
    [LAT, LON], tooltip="Input outlet",
    popup=f"Input outlet<br>{LAT:.6f}, {LON:.6f}",
    icon=folium.Icon(color="red")
).add_to(m)

folium.CircleMarker(
    [snapped.y, snapped.x], radius=7, color="orange", fill=True,
    tooltip="NLDI hydrolocation", popup=f"COMID: {COMID}"
).add_to(m)

b = basin.total_bounds
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
folium.LayerControl().add_to(m)

map_path = OUTPUTS / "catchment_map.html"
m.save(map_path)
print("Saved:", map_path)
m


## 7. Export GIS layers

These files can be opened in QGIS or ArcGIS.


In [ ]:
basin.to_file(OUTPUTS/"catchment.geojson", driver="GeoJSON")

outlet = gpd.GeoDataFrame(
    {"latitude":[LAT], "longitude":[LON], "comid":[COMID]},
    geometry=[Point(LON, LAT)], crs="EPSG:4326"
)
outlet.to_file(OUTPUTS/"outlet.geojson", driver="GeoJSON")

if streams is not None:
    streams.to_file(OUTPUTS/"upstream_streams.geojson", driver="GeoJSON")

print("GIS outputs written to", OUTPUTS)


## 8. PRISM catchment-area-weighted extraction

This section downloads official daily 4-km PRISM grids. Each downloaded ZIP is cached, so rerunning the notebook avoids re-downloading an existing grid.

The spatial average is weighted by the area of each PRISM grid cell that lies inside the catchment. Intersection areas are calculated in CONUS Albers Equal Area (EPSG:5070).

**Important:** a 26-year run with 3 variables requires roughly 28,000 daily grid requests. Test one month first.


In [ ]:
def prism_zip_path(variable, date):
    ds = pd.Timestamp(date).strftime("%Y%m%d")
    return CACHE / f"{variable}_{ds}.zip"

def download_prism(variable, date, retries=3):
    ds = pd.Timestamp(date).strftime("%Y%m%d")
    path = prism_zip_path(variable, date)
    if path.exists() and path.stat().st_size > 1000:
        return path

    url = f"{PRISM}/us/4km/{variable}/{ds}"
    last_error = None
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=180)
            r.raise_for_status()
            path.write_bytes(r.content)
            return path
        except Exception as e:
            last_error = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"PRISM download failed: {variable} {ds}: {last_error}")

def extract_prism_weighted_mean(zip_path, basin_wgs84):
    temp_dir = Path("/content/prism_temp")
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir()

    try:
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(temp_dir)

        bils = list(temp_dir.rglob("*.bil"))
        if not bils:
            raise RuntimeError(f"No .bil raster found in {zip_path.name}")

        with rasterio.open(bils[0]) as src:
            # Reproject basin to raster CRS to determine candidate raster window.
            basin_src = basin_wgs84.to_crs(src.crs)
            geom = basin_src.geometry.unary_union
            minx, miny, maxx, maxy = geom.bounds
            window = rasterio.windows.from_bounds(minx, miny, maxx, maxy, src.transform)
            window = window.round_offsets().round_lengths()
            data = src.read(1, window=window, masked=True)
            transform = src.window_transform(window)

            # Build candidate cell polygons in raster CRS, then calculate exact
            # intersection areas in an equal-area CRS.
            rows, cols = np.where(~np.ma.getmaskarray(data))
            values, polygons = [], []
            for rr, cc in zip(rows, cols):
                val = float(data[rr, cc])
                if not np.isfinite(val):
                    continue
                left, top = xy(transform, rr, cc, offset="ul")
                right, bottom = xy(transform, rr, cc, offset="lr")
                polygons.append(box(min(left,right), min(bottom,top), max(left,right), max(bottom,top)))
                values.append(val)

            if not polygons:
                return np.nan

            cells = gpd.GeoDataFrame({"value": values}, geometry=polygons, crs=src.crs)
            cells = cells[cells.intersects(geom)].copy()
            if cells.empty:
                return np.nan

            cells_ea = cells.to_crs("EPSG:5070")
            basin_union_ea = basin_wgs84.to_crs("EPSG:5070").geometry.unary_union
            weights = cells_ea.geometry.intersection(basin_union_ea).area.to_numpy()
            vals = cells_ea["value"].to_numpy()

            valid = (weights > 0) & np.isfinite(vals)
            if not valid.any():
                return np.nan

            return float(np.average(vals[valid], weights=weights[valid]))
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)


In [ ]:
dates = pd.date_range(START_DATE, END_DATE, freq="D")
print(f"{len(dates):,} days × {len(PRISM_VARIABLES)} variables = {len(dates)*len(PRISM_VARIABLES):,} grid requests maximum")
print("Cached grids are reused.")

records = []
errors = []

for date in tqdm(dates, desc="PRISM days"):
    row = {"Date": date}
    for variable in PRISM_VARIABLES:
        try:
            zp = download_prism(variable, date)
            row[variable] = extract_prism_weighted_mean(zp, basin)
        except Exception as e:
            row[variable] = np.nan
            errors.append((str(date.date()), variable, str(e)))
    records.append(row)

df_prism = pd.DataFrame(records).rename(columns={
    "ppt":"Prcp_mm",
    "tmin":"Tmin_C",
    "tmax":"Tmax_C",
    "tmean":"Tmean_C"
})

if "Tmin_C" in df_prism and "Tmax_C" in df_prism and "Tmean_C" not in df_prism:
    df_prism["Tmean_C"] = (df_prism["Tmin_C"] + df_prism["Tmax_C"]) / 2

display(df_prism.head())
print("\nMissing values:")
display(df_prism.isna().sum())

if errors:
    print(f"Warnings/errors: {len(errors)}. First five:")
    for e in errors[:5]:
        print(e)


## 9. Plot meteorology


In [ ]:
if "Prcp_mm" in df_prism:
    plt.figure(figsize=(14,4))
    plt.bar(df_prism["Date"], df_prism["Prcp_mm"], width=1)
    plt.xlabel("Date")
    plt.ylabel("Precipitation (mm)")
    plt.title("Catchment-Area-Weighted PRISM Precipitation")
    plt.tight_layout()
    plt.show()

if {"Tmin_C","Tmax_C"}.issubset(df_prism.columns):
    plt.figure(figsize=(14,4))
    plt.plot(df_prism["Date"], df_prism["Tmin_C"], label="Tmin")
    plt.plot(df_prism["Date"], df_prism["Tmax_C"], label="Tmax")
    plt.xlabel("Date")
    plt.ylabel("Temperature (°C)")
    plt.title("Catchment-Area-Weighted PRISM Temperature")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Save and download results


In [ ]:
from google.colab import files

csv_name = f"PRISM_Catchment_Average_{START_DATE}_to_{END_DATE}.csv"
csv_path = OUTPUTS / csv_name
df_prism.to_csv(csv_path, index=False)

# Bundle generated outputs
bundle = "/content/catchment_climate_outputs.zip"
shutil.make_archive(bundle.replace(".zip",""), "zip", OUTPUTS)

print("Created:", bundle)
files.download(bundle)


## Notes for research use

- Visually verify the outlet and basin before using the results.
- NLDI/NHDPlus delineation and a high-resolution DEM delineation are not identical methodologies.
- PRISM values are gridded estimates, not direct station observations.
- Record the retrieval date and software version for reproducibility.
- Cite the official USGS NLDI/NHDPlus and PRISM data/services in publications.
